# 🛡️ SmartAPD - PPE Detection Model Training

**Classes**: Hardhat, NO-Hardhat, Mask, NO-Mask, Safety Vest, NO-Safety Vest, Person, Safety Cone, machinery, vehicle

## 🚀 Cara Pakai:
1. **Runtime** → **Change runtime type** → **T4 GPU**
2. **Runtime** → **Run all**
3. Tunggu ~30-60 menit
4. Download model dari output

In [ ]:
# Check GPU
!nvidia-smi -L
import torch
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Install
!pip install ultralytics==8.1.29 kagglehub -qq
import ultralytics
print(f"Ultralytics: {ultralytics.__version__}")

In [ ]:
# Download dataset from Kaggle (NO API KEY NEEDED)
import kagglehub
import os

path = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")
print(f"Downloaded to: {path}")

# Find dataset folder
dataset_location = None
for root, dirs, files in os.walk(path):
    if 'train' in dirs and 'valid' in dirs:
        dataset_location = root
        break
if not dataset_location:
    dataset_location = f"{path}/css-data"

print(f"\n✅ Dataset: {dataset_location}")
!ls {dataset_location}

In [ ]:
# Create data.yaml
import yaml

CLASSES = ['Hardhat', 'Mask', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 
           'Person', 'Safety Cone', 'Safety Vest', 'machinery', 'vehicle']

data_yaml = {
    'train': f'{dataset_location}/train',
    'val': f'{dataset_location}/valid',
    'test': f'{dataset_location}/test',
    'nc': len(CLASSES),
    'names': CLASSES
}

with open('data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print("✅ data.yaml created")
!cat data.yaml

In [ ]:
%%time
# Train YOLOv8
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

model = YOLO('yolov8s.pt')

results = model.train(
    data='data.yaml',
    epochs=80,
    batch=16,
    imgsz=640,
    patience=25,
    name='smartapd_ppe',
    exist_ok=True,
    device=0,
)

print("\n✅ Training complete!")

In [ ]:
# Copy best model
import shutil
shutil.copy('runs/detect/smartapd_ppe/weights/best.pt', '/content/smartapd_ppe_model.pt')
print("✅ Model: /content/smartapd_ppe_model.pt")

import os
size = os.path.getsize('/content/smartapd_ppe_model.pt') / (1024*1024)
print(f"📦 Size: {size:.2f} MB")

In [ ]:
# Download model
from google.colab import files
files.download('/content/smartapd_ppe_model.pt')

## ✅ Done!
Pindahkan `smartapd_ppe_model.pt` ke folder `ai-engine/` di project SmartAPD